# MLflow: трекинг экспериментов и воспроизводимость

**Проект «ИИ для учёных».** Практический блокнот к разделу «Инструменты».

## Что делает этот инструмент

MLflow — система управления жизненным циклом ML-экспериментов. Она решает типичную проблему исследователя: «Я запускал модель с разными параметрами, получил несколько результатов — но теперь не помню, какой прогон дал лучшее качество и с какими настройками».

MLflow автоматически записывает в базу данных каждый прогон: гиперпараметры, метрики, артефакты (графики, модели). Любой эксперимент можно воспроизвести точно — без записных книжек и таблиц Excel.

В этом блокноте мы пройдём полный цикл: подбор гиперпараметров с логированием каждого прогона → сравнение всех прогонов в pandas → выбор лучшей модели → загрузка лучшей модели из реестра. Всё работает локально в Colab без запуска UI-сервера. Расчётное время прохождения — около пяти минут.

## Ключевые понятия MLflow

- **Run (прогон)** — один запуск обучения с фиксированными гиперпараметрами. Каждому присваивается уникальный ID.
- **Experiment (эксперимент)** — именованная группа прогонов, объединённых общей задачей.
- **Params** — гиперпараметры, которые вы задаёте до обучения (глубина дерева, число итераций).
- **Metrics** — числовые показатели качества, вычисленные после обучения (R², MAE).
- **Artifacts** — файлы, созданные в ходе прогона: графики, обученные модели, таблицы.
- **Model Registry** — реестр версий моделей с тегами (staging, production). Позволяет управлять жизненным циклом модели от прототипа до деплоя.
- **mlruns/** — локальная папка, куда MLflow по умолчанию сохраняет всё. В Colab запускать UI необязательно: всё доступно через Python API.

## 1. Установка библиотек

In [ ]:
!pip install -q mlflow scikit-learn pandas matplotlib

## 2. Единый стиль графиков

Графики оформляются в едином визуальном стиле сайта «ИИ для учёных». Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py`.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",
    "node_fill":  "#eef1f6",
    "node_text":  "#0e1726",
    "edge":       "#4d5e78",
    "grid":       "#dce2ec",
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Данные и постановка задачи

Используем встроенный набор о диабете (diabetes dataset из scikit-learn): 442 пациента, 10 числовых признаков, целевая величина — количественный показатель прогрессирования болезни через год. Это классическая задача регрессии с небольшим объёмом данных, удобная для демонстрации.

Задача: перебрать несколько вариантов гиперпараметров GradientBoostingRegressor, залогировать каждый прогон в MLflow и выбрать лучший.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split

data = load_diabetes(as_frame=True)
X, y = data.data, data.target
feature_names = list(X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Набор данных: {X.shape[0]} наблюдений, {X.shape[1]} признаков")
print(f"Обучающая выборка: {X_train.shape[0]}, тестовая: {X_test.shape[0]}")
X.head(3)

## 4. Настройка MLflow

Указываем MLflow хранить все данные в локальной папке `./mlruns`. В Colab UI-сервер запускать не нужно — всё доступно через Python API `mlflow.search_runs()`.

Создаём именованный эксперимент. Все прогоны ниже войдут в него.

In [ ]:
import mlflow
import mlflow.sklearn

# Все данные сохраняются локально в ./mlruns — UI не нужен
mlflow.set_tracking_uri("./mlruns")

EXPERIMENT_NAME = "diabetes-gbm-tuning"

# Если эксперимент уже существует — используем его; иначе создаём новый
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
else:
    experiment_id = experiment.experiment_id

mlflow.set_experiment(EXPERIMENT_NAME)
print(f"Эксперимент: '{EXPERIMENT_NAME}'  (id={experiment_id})")
print(f"Данные сохраняются в: ./mlruns/")

## 5. Цикл обучения с логированием

Перебираем сетку гиперпараметров. Каждый вариант — отдельный прогон MLflow. Внутри каждого прогона логируем:
- **params**: гиперпараметры модели,
- **metrics**: R² и MAE на тестовой выборке,
- **артефакт**: график «прогноз vs. факт» в формате PNG,
- **модель**: сохраняется через `mlflow.sklearn.log_model` для последующей загрузки.

Блок `with mlflow.start_run()` автоматически закрывает прогон даже при ошибке.

In [ ]:
import matplotlib.pyplot as plt
import tempfile, os
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error

# Сетка гиперпараметров для перебора
param_grid = [
    {"n_estimators": 50,  "learning_rate": 0.1,  "max_depth": 2},
    {"n_estimators": 100, "learning_rate": 0.1,  "max_depth": 3},
    {"n_estimators": 200, "learning_rate": 0.05, "max_depth": 3},
    {"n_estimators": 100, "learning_rate": 0.2,  "max_depth": 4},
    {"n_estimators": 200, "learning_rate": 0.1,  "max_depth": 4},
    {"n_estimators": 50,  "learning_rate": 0.3,  "max_depth": 2},
]

print(f"Запускаем {len(param_grid)} прогонов...\n")

for params in param_grid:
    with mlflow.start_run():
        # 1. Логируем гиперпараметры
        mlflow.log_params(params)

        # 2. Обучаем модель
        model = GradientBoostingRegressor(random_state=42, **params)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # 3. Считаем и логируем метрики
        r2  = r2_score(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)
        mlflow.log_metric("r2",  r2)
        mlflow.log_metric("mae", mae)

        # 4. Сохраняем артефакт: график прогноз vs. факт
        fig, ax = plt.subplots(figsize=(5, 4))
        ax.scatter(y_test, y_pred, s=20, alpha=0.5,
                   color=VIZ["series"][0])
        lims = [y_test.min(), y_test.max()]
        ax.plot(lims, lims, color=VIZ["series"][2],
                linestyle="--", label="Идеал")
        ax.set_xlabel("Истинное значение")
        ax.set_ylabel("Предсказанное значение")
        ax.set_title(f"n={params['n_estimators']}, lr={params['learning_rate']}, "
                     f"depth={params['max_depth']}\nR²={r2:.3f}, MAE={mae:.1f}")
        ax.legend()
        fig.tight_layout()

        with tempfile.TemporaryDirectory() as tmpdir:
            fig_path = os.path.join(tmpdir, "pred_vs_true.png")
            fig.savefig(fig_path, dpi=100)
            mlflow.log_artifact(fig_path)
        plt.close(fig)

        # 5. Сохраняем модель (для последующей загрузки)
        mlflow.sklearn.log_model(model, artifact_path="model")

        print(f"  n={params['n_estimators']:3d} | lr={params['learning_rate']:.2f} | "
              f"depth={params['max_depth']} | R²={r2:.3f} | MAE={mae:.1f}")

print("\nВсе прогоны завершены и залогированы.")

## 6. Сравнение прогонов через pandas

`mlflow.search_runs()` возвращает все прогоны эксперимента в виде pandas DataFrame. Это позволяет анализировать результаты программно: сортировать, фильтровать, строить графики.

In [ ]:
# Загружаем все прогоны эксперимента в DataFrame
runs_df = mlflow.search_runs(experiment_names=[EXPERIMENT_NAME])

# Выбираем нужные столбцы: параметры и метрики
cols = [
    "run_id",
    "params.n_estimators",
    "params.learning_rate",
    "params.max_depth",
    "metrics.r2",
    "metrics.mae",
]
summary = runs_df[cols].copy()
summary.columns = ["run_id", "n_estimators", "learning_rate", "max_depth", "R²", "MAE"]
summary["n_estimators"]  = summary["n_estimators"].astype(int)
summary["learning_rate"] = summary["learning_rate"].astype(float)
summary["max_depth"]     = summary["max_depth"].astype(int)
summary["R²"]            = summary["R²"].round(4)
summary["MAE"]           = summary["MAE"].round(2)

# Сортируем по убыванию R²
summary = summary.sort_values("R²", ascending=False).reset_index(drop=True)
print("Все прогоны, отсортированные по R² (лучший сверху):")
summary

### Визуализация: метрика vs. гиперпараметр

Построим два графика, которые помогают понять, как гиперпараметры влияют на качество модели. Это типичный анализ после перебора параметров.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.2))

# --- Левый: R² vs. n_estimators, цвет — learning_rate ---
lr_vals    = summary["learning_rate"].unique()
lr_palette = get_palette(len(lr_vals))

for lr, color in zip(sorted(lr_vals), lr_palette):
    subset = summary[summary["learning_rate"] == lr].sort_values("n_estimators")
    axes[0].scatter(
        subset["n_estimators"], subset["R²"],
        color=color, s=90, zorder=3, label=f"lr={lr}"
    )
    if len(subset) > 1:
        axes[0].plot(subset["n_estimators"], subset["R²"],
                     color=color, linewidth=1.2, alpha=0.6)

axes[0].set_title("R² vs. число деревьев")
axes[0].set_xlabel("n_estimators")
axes[0].set_ylabel("R² (тестовая выборка)")
axes[0].legend(title="learning_rate", loc="lower right")

# --- Правый: MAE vs. max_depth (группировка по n_estimators) ---
depth_vals = sorted(summary["max_depth"].unique())
x_pos = np.arange(len(depth_vals))
width = 0.25
n_est_vals = sorted(summary["n_estimators"].unique())

for i, n_est in enumerate(n_est_vals):
    subset = summary[summary["n_estimators"] == n_est].set_index("max_depth")
    mae_vals = [subset.loc[d, "MAE"] if d in subset.index else float("nan")
                for d in depth_vals]
    axes[1].bar(
        x_pos + i * width, mae_vals, width,
        color=get_palette(len(n_est_vals))[i],
        label=f"n={n_est}", alpha=0.85,
    )

axes[1].set_xticks(x_pos + width)
axes[1].set_xticklabels([f"depth={d}" for d in depth_vals])
axes[1].set_title("MAE vs. глубина дерева")
axes[1].set_ylabel("MAE (тестовая выборка)")
axes[1].legend(title="n_estimators", loc="upper right")

fig.tight_layout()
plt.show()

**Как читать эти графики.**

Левый: каждая точка — прогон MLflow. Если R² растёт с числом деревьев — добавление деревьев улучшает прогноз. Если кривая выходит на плато — дальнейшее увеличение не даёт выигрыша.

Правый: группы столбцов по глубине дерева, цвет — число деревьев. Более глубокие деревья при малом их числе могут переобучаться (MAE растёт). Оптимальная комбинация — минимальный столбец.

## 7. Лучший прогон: выбор и загрузка модели

Находим прогон с максимальным R², затем загружаем сохранённую модель прямо из хранилища MLflow — без повторного обучения. Это и есть воспроизводимость: любой прогон можно восстановить по его run_id.

In [ ]:
# Лучший прогон по R²
best_row = summary.iloc[0]
best_run_id = best_row["run_id"]

print(f"Лучший прогон:")
print(f"  run_id:        {best_run_id}")
print(f"  n_estimators:  {best_row['n_estimators']}")
print(f"  learning_rate: {best_row['learning_rate']}")
print(f"  max_depth:     {best_row['max_depth']}")
print(f"  R²:            {best_row['R²']}")
print(f"  MAE:           {best_row['MAE']}")

# Загружаем сохранённую модель из артефактов MLflow
model_uri = f"runs:/{best_run_id}/model"
best_model = mlflow.sklearn.load_model(model_uri)
print(f"\nМодель загружена из {model_uri}")

# Проверяем: предсказания совпадают с залогированными метриками
y_pred_check = best_model.predict(X_test)
r2_check  = r2_score(y_test, y_pred_check)
mae_check = mean_absolute_error(y_test, y_pred_check)
print(f"Проверка: R²={r2_check:.4f}, MAE={mae_check:.2f}  (должны совпасть с таблицей)")

## 8. Концепция реестра моделей (Model Registry)

В этом блокноте мы работали с трекингом прогонов. MLflow также предоставляет **Model Registry** — централизованный каталог версий моделей. После регистрации модели можно переводить между стадиями: `None → Staging → Production → Archived`.

В Colab Model Registry работает, но требует настроенного backend-сервера с базой данных (SQLite или PostgreSQL). Ниже показано, как выглядит регистрация — для ознакомления.

In [ ]:
# Регистрация лучшей модели в Model Registry
# В файловом backend (./mlruns) это работает через MlflowClient.

from mlflow.tracking import MlflowClient

client = MlflowClient(tracking_uri="./mlruns")

MODEL_NAME = "diabetes-gbm-best"

try:
    # Регистрируем модель из лучшего прогона
    result = mlflow.register_model(
        model_uri=f"runs:/{best_run_id}/model",
        name=MODEL_NAME,
    )
    print(f"Модель зарегистрирована: '{MODEL_NAME}', версия {result.version}")

    # Смотрим список версий
    versions = client.search_model_versions(f"name='{MODEL_NAME}'")
    for v in versions:
        print(f"  Версия {v.version}: run_id={v.run_id[:8]}..., "
              f"статус={v.status}")

except Exception as e:
    # В некоторых средах файловый backend не поддерживает Registry
    print(f"Model Registry недоступен в данной конфигурации: {e}")
    print("В production: используйте mlflow server --backend-store-uri sqlite:///mlflow.db")

## 9. Интерпретация результатов

- **Таблица прогонов**: каждая строка — воспроизводимый эксперимент с фиксированными параметрами и метриками. Если коллега хочет повторить прогон — достаточно знать `run_id`.
- **График R² vs. n_estimators**: помогает понять, когда добавление деревьев перестаёт улучшать качество. Плато — сигнал остановиться.
- **Лучший прогон**: `mlflow.sklearn.load_model` загружает точно ту модель, которая была обучена в этом прогоне — с теми же весами.
- **Зачем это нужно**: в долгих вычислительных экспериментах (дни/недели) журнал MLflow — единственный надёжный способ не потерять результаты и уметь воспроизвести лучшую версию.

Типичная ошибка: не логировать версию данных. Добавьте `mlflow.log_param("data_hash", hash_value)` — тогда прогоны с разными версиями данных не перепутаются.

## 10. Попробуйте на своих данных

1. Замените `load_diabetes()` своим датасетом (`pd.read_csv(...)`).
2. Укажите другой `EXPERIMENT_NAME` — прогоны не перемешаются.
3. Расширьте `param_grid` своими значениями или добавьте другие модели (`RandomForestRegressor`, `Ridge`).
4. Добавьте в каждый прогон дополнительные метрики или артефакты (например, матрицу важности признаков).
5. Для просмотра UI запустите в терминале (не в Colab): `mlflow ui --backend-store-uri ./mlruns`.

In [ ]:
# --- Шаблон: добавить свой датасет и модель ---
# import pandas as pd
# from sklearn.model_selection import train_test_split
#
# df = pd.read_csv("my_data.csv")
# target_col = "целевая_величина"
# X = df.drop(columns=[target_col])
# y = df[target_col]
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
#
# MY_PARAM_GRID = [
#     {"n_estimators": 100, "learning_rate": 0.1, "max_depth": 3},
#     # ... другие варианты
# ]
#
# mlflow.set_experiment("my-experiment")
# for params in MY_PARAM_GRID:
#     with mlflow.start_run():
#         mlflow.log_params(params)
#         model = GradientBoostingRegressor(**params, random_state=42)
#         model.fit(X_train, y_train)
#         y_pred = model.predict(X_test)
#         mlflow.log_metric("r2",  r2_score(y_test, y_pred))
#         mlflow.log_metric("mae", mean_absolute_error(y_test, y_pred))
#         mlflow.sklearn.log_model(model, artifact_path="model")
print("Шаблон готов. Раскомментируйте, подставьте свои данные и перезапустите.")

## Что дальше

- Официальная документация MLflow: https://mlflow.org/docs/latest/index.html
- Quickstart (Python API): https://mlflow.org/docs/latest/getting-started/intro-quickstart/index.html
- Model Registry: https://mlflow.org/docs/latest/model-registry.html
- Запуск сервера с SQLite-backend для полного Registry: `mlflow server --backend-store-uri sqlite:///mlflow.db --default-artifact-root ./artifacts`
- Интеграция с другими фреймворками: `mlflow.pytorch`, `mlflow.tensorflow`, `mlflow.xgboost` — те же принципы, автологирование через `mlflow.autolog()`.
- Для командных проектов: разворачивайте MLflow Tracking Server на общем сервере — тогда все участники видят прогоны друг друга.